# SAM-Audio Large MP3 Separation

Upload or choose an MP3, describe the sound you want to isolate, and run `facebook/sam-audio-large`.

This model is gated on Hugging Face. Request access to `facebook/sam-audio-large` before running the model-loading cell.

## Memory Notes

The checkpoint file is about 14.9 GB. This notebook uses the Large checkpoint, CUDA BF16, and audio-only mode. It deletes the vision encoder before moving the model to GPU, but keeps rankers and the span predictor available. It is intended for 30-second clips and does not do chunking.

In [ ]:
# Run once per fresh, dedicated runtime. SAM-Audio has a broad dependency tree.
# Some transitive packages downgrade shared Colab/Kaggle packages, so avoid using this runtime for JAX/TensorFlow/Google SDK work.
%pip install -q --upgrade pip

# This notebook does not use TensorFlow/JAX. Removing them avoids protobuf import crashes after SAM-Audio installs its audio codec stack.
%pip uninstall -y -q tensorflow tensorflow-cpu tensorflow-text tensorflow-datasets tensorflow-metadata tf-keras keras jax jaxlib

%pip install -q --no-warn-conflicts "sam_audio @ git+https://github.com/facebookresearch/sam-audio.git"

# SAM-Audio currently expects the Transformers 4.x / Hugging Face Hub 0.x API family.
# Do not upgrade protobuf here; descript-audiotools, used by the audio codec stack, requires protobuf <3.20.
%pip install -q --upgrade --no-warn-conflicts "transformers>=4.54,<5" "huggingface_hub>=0.34,<1.0"

## Restart Runtime

Run the next cell after the setup cell, then continue from the dependency diagnostics cell after the runtime restarts. This is required because SAM-Audio's install can replace NumPy while the current Python process is still running.

In [ ]:
import os

print("Restarting runtime so NumPy binary modules reload cleanly...")
os.kill(os.getpid(), 9)

In [ ]:
# Optional dependency diagnostics. Leave this off for the normal SAM-Audio workflow.
# Conflicts with unrelated hosted-runtime packages are expected.
import subprocess
import sys

RUN_PIP_CHECK = False

if RUN_PIP_CHECK:
    subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
else:
    print("Skipped pip check. Set RUN_PIP_CHECK = True to inspect hosted-runtime dependency conflicts.")

## Hugging Face Login

Run this if the model-loading cell cannot access the gated checkpoint. You need an accepted access request for `facebook/sam-audio-large`.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import gc
import os
import shutil
import subprocess
import types
import warnings
import zipfile
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)

import torch
import torchaudio
from pydub import AudioSegment
from sam_audio import SAMAudio, SAMAudioProcessor

MODEL_ID = "facebook/sam-audio-large"
OUTPUT_DIR = Path("sam_audio_outputs")
ZIP_PATH = Path("sam_audio_outputs.zip")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this BF16 SAM-Audio Large notebook.")

DEVICE = "cuda"
DTYPE = torch.bfloat16

print(f"Device: {DEVICE}")
print(f"Dtype: {DTYPE}")

In [ ]:
def ensure_ffmpeg() -> None:
    if shutil.which("ffmpeg"):
        return

    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is required for MP3 conversion.")


ensure_ffmpeg()

In [ ]:
def disable_vision_encoder_for_audio_only(model):
    if not hasattr(model, "vision_encoder"):
        raise AttributeError("Expected SAM-Audio model to have a vision_encoder.")

    vision_dim = getattr(model.vision_encoder, "dim", None)
    if vision_dim is None:
        raise AttributeError("Could not read model.vision_encoder.dim.")

    del model.vision_encoder
    model._vision_encoder_dim = vision_dim

    def _get_video_features_audio_only(self, video, audio_features):
        if video is not None:
            raise ValueError("This notebook is audio-only; video inputs are disabled.")
        batch_size, time_steps, _ = audio_features.shape
        return audio_features.new_zeros(batch_size, self._vision_encoder_dim, time_steps)

    model._get_video_features = types.MethodType(_get_video_features_audio_only, model)
    gc.collect()
    torch.cuda.empty_cache()
    return model


model = SAMAudio.from_pretrained(MODEL_ID, proxies=None, resume_download=False)
model = disable_vision_encoder_for_audio_only(model)
model = model.to(DEVICE, DTYPE).eval()
processor = SAMAudioProcessor.from_pretrained(MODEL_ID)

print("Audio-only mode: vision_encoder disabled before CUDA load.")
print(f"Rankers kept: visual_ranker={getattr(model, 'visual_ranker', None) is not None}, text_ranker={getattr(model, 'text_ranker', None) is not None}")
print(f"Span predictor kept: {hasattr(model, 'span_predictor') and getattr(model, 'span_predictor', None) is not None}")

print(f"CUDA allocated after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"CUDA reserved after load:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## Upload Audio

In [ ]:
def upload_or_choose_mp3() -> Path:
    try:
        from google.colab import files
    except ModuleNotFoundError:
        audio_path = input("Paste the path to your MP3/audio file: ").strip()
        if not audio_path:
            raise ValueError("No audio path provided.")
        return Path(audio_path).expanduser().resolve()

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    first_name = next(iter(uploaded))
    return Path(first_name).resolve()


AUDIO_PATH = upload_or_choose_mp3()
print(AUDIO_PATH)

## Target Sound Prompt

Use a short noun phrase or verb phrase. Examples: `footsteps`, `car engine`, `wind`, `dog barking`, `piano`, `man speaking`.

In [ ]:
DESCRIPTION = "footsteps"
MAX_AUDIO_SECONDS = 35
PREDICT_SPANS = False
RERANKING_CANDIDATES = 1

Optional span anchors can help when you know where the target sound occurs. Use `+` for present and `-` for absent.

Example:

```python
ANCHORS = [["+", 6.3, 7.0], ["-", 0.0, 1.0]]
```

In [ ]:
ANCHORS = None

In [ ]:
def tensor_from_result(value):
    if isinstance(value, (list, tuple)):
        return value[0]
    return value


def save_waveform(path: Path, waveform: torch.Tensor, sample_rate: int) -> None:
    waveform = waveform.detach().to(torch.float32).cpu()
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    torchaudio.save(str(path), waveform, sample_rate)


def wav_to_mp3(wav_path: Path, mp3_path: Path) -> None:
    AudioSegment.from_file(wav_path).export(mp3_path, format="mp3", bitrate="192k")


def zip_outputs(paths: list[Path], zip_path: Path = ZIP_PATH) -> Path:
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for path in paths:
            zip_file.write(path, arcname=path.name)

    return zip_path

In [ ]:
@torch.inference_mode()
def separate_audio(
    audio_path: Path,
    description: str,
    anchors=None,
    predict_spans: bool = False,
    reranking_candidates: int = 1,
):
    duration_seconds = AudioSegment.from_file(audio_path).duration_seconds
    if duration_seconds > MAX_AUDIO_SECONDS:
        raise ValueError(
            f"This notebook is configured for ~30s clips. Got {duration_seconds:.1f}s. "
            "Use split_mp3_30s_overlap.ipynb first or increase MAX_AUDIO_SECONDS."
        )

    processor_kwargs = {
        "audios": [str(audio_path)],
        "descriptions": [description],
    }
    if anchors:
        processor_kwargs["anchors"] = [anchors]

    with torch.autocast(device_type=DEVICE, dtype=DTYPE):
        inputs = processor(**processor_kwargs).to(DEVICE)
        return model.separate(
            inputs,
            predict_spans=predict_spans,
            reranking_candidates=reranking_candidates,
        )


if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.cuda.reset_peak_memory_stats()
result = separate_audio(
    AUDIO_PATH,
    DESCRIPTION,
    anchors=ANCHORS,
    predict_spans=PREDICT_SPANS,
    reranking_candidates=RERANKING_CANDIDATES,
)

sample_rate = processor.audio_sampling_rate
target_wav = OUTPUT_DIR / "target.wav"
residual_wav = OUTPUT_DIR / "residual.wav"
target_mp3 = OUTPUT_DIR / "target.mp3"
residual_mp3 = OUTPUT_DIR / "residual.mp3"

save_waveform(target_wav, tensor_from_result(result.target), sample_rate)
save_waveform(residual_wav, tensor_from_result(result.residual), sample_rate)
wav_to_mp3(target_wav, target_mp3)
wav_to_mp3(residual_wav, residual_mp3)

zip_path = zip_outputs([target_wav, residual_wav, target_mp3, residual_mp3])

print(f"Target sound: {DESCRIPTION}")
print(f"Saved: {target_wav.resolve()}")
print(f"Saved: {residual_wav.resolve()}")
print(f"Saved: {target_mp3.resolve()}")
print(f"Saved: {residual_mp3.resolve()}")
print(f"Zip:   {zip_path.resolve()}")

print(f"Peak CUDA allocated during separation: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Peak CUDA reserved during separation:  {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

In [ ]:
try:
    from google.colab import files
except ModuleNotFoundError:
    print(f"Download or use the zip file here: {ZIP_PATH.resolve()}")
else:
    files.download(str(ZIP_PATH))